Tienes la variable objetivo (ratio), pero para que el modelo aprenda necesita predictores. Ahora mismo tienes tipo de fase, material, familia, cantidad, dedicación... Hay que repasar qué features tienes y cuáles derivar (por ejemplo, el rendimiento histórico de cada familia de fase en cada máquina, que suele ser lo más predictivo). Ese ejercicio decide si tu proyecto es rico, y responde tu vieja preocupación de "si da para IA".

Tu problema es de idoneidad fase-máquina. El modelo predice el ratio a partir de características disponibles antes de ejecutar la fase. Esa última parte es la regla de oro — todo lo que uses como predictor tiene que ser información que existiría en el momento de decidir a qué máquina asignar la fase. Si usas algo que solo se conoce después de ejecutarla, es leakage y el modelo hace trampa.

Leakage es cuando tu modelo aprende usando información que no estaría disponible en el momento real de hacer la predicción. El modelo parece funcionar de maravilla en tus pruebas, pero es un espejismo: está "haciendo trampa" usando datos que en la vida real no tendría. Cuando lo pusieras a funcionar de verdad, fracasaría.

La pregunta que define todo: en el momento en que el planificador decide a qué máquina mandar una fase, ¿qué sabe y qué no sabe? Todo lo que sabe → puede ser predictor. Todo lo que solo se conoce después de ejecutar la fase → es leakage, prohibido.

El caso más obvio en tu proyecto

Tu variable objetivo es el ratio = tiempo_real / tiempo_teórico. El tiempo_real se conoce después de ejecutar la fase. Así que:

Usar tiempo_real_efectivo como predictor sería leakage total y absurdo — estarías usando parte de la respuesta para predecir la respuesta. Es como predecir si un alumno aprobará usando su nota final. El modelo tendría 100% de acierto y sería completamente inútil.

Por la misma razón, dedicacion_media, n_imputaciones, todos los MinutosReal_* → leakage. Todos se conocen solo tras ejecutar. En el momento de asignar, no existen aún.

La features saldrán de lo que ERP/MES ya registran, no de instrumentar la planta

### Predicción de la desviación temporal

In [14]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pyarrow as pa

In [15]:
df_entrenamiento = pd.read_parquet("res//procesed/df_entrenamiento.parquet")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)   
display(df_entrenamiento.head(1))

,fecha_fase,IDMOTask,smOrdenId,smOrdenNme,smFaseNme,Description,IDUsualTask,MaquinaDominante,NombreMaquina,CodigoGrupo,NombreGrupo,smMaterialId,Descripción Artículo,CodProductLine,Nombre Línea,CodProductFamily,Nombre Família,ProductEntry,MaterialExit,Quantity,tiempo_real_efectivo,tiempo_previsto,ratio
0,2025-10-02 03:12:05.010,ee76d685-a4cc-47d4-b418-6e33d2ae2921,OF2509545,Portacápsulas F704501049704(150713),TPP,Mecanitzat al Torn Puma Petit,29783995-2f56-48be-9664-461e6f0b402a,TN09,TPP DOOSAN PUMA 240 (TPP) Torn C.N.C,TPU,Tornos Puma Cobre,PE1JNCX0012,Elektrodenaufnamhe F704501049704(DS086228),P,Portalectrodes,PE,Portalectrodes,0.0,0.0,3.0,41.76667,21.0,1.988889


## OHE
Es necesario trasformar todas las varibles de categorias. 

In [16]:
df_features = df_entrenamiento
df_features = df_features.drop(columns={"MaterialExit", "ProductEntry", "smMaterialId","IDUsualTask", "tiempo_real_efectivo", "smOrdenNme", "Description", "NombreMaquina", "NombreGrupo", "Descripción Artículo", "Nombre Línea", "Nombre Família"}
                               ).rename(columns={"smFaseNme":"TipoFase", "MaquinaDominante": "CodigoMaquina", "CodigoGrupo": "Familia Maquinas", "CodProductLine": "Línea Producto", "CodProductFamily": "Família Producto", "tiempo_previsto": "TiempoPrevisto"})
display(df_features.head(0))
df_features.shape


,fecha_fase,IDMOTask,smOrdenId,TipoFase,CodigoMaquina,Familia Maquinas,Línea Producto,Família Producto,Quantity,TiempoPrevisto,ratio


(14223, 11)

In [17]:
cols_texto = df_features.select_dtypes(include=["object", "category"]).columns

resumen = pd.DataFrame({
    "n_unicos": df_features[cols_texto].nunique(),
    "pct_unicos": (df_features[cols_texto].nunique() / len(df_features) * 100).round(2),
    "n_nulos": df_features[cols_texto].isna().sum(),
    "ejemplo": [df_features[c].dropna().iloc[0] for c in cols_texto],
})#.sort_values("n_unicos")

display(resumen)

,n_unicos,pct_unicos,n_nulos,ejemplo
IDMOTask,14223,100.00,0,ee76d685-a4cc-47d4-b418-6e33d2ae2921
smOrdenId,6386,44.90,0,OF2509545
TipoFase,38,0.27,0,TPP
CodigoMaquina,23,0.16,0,TN09
Familia Maquinas,9,0.06,0,TPU
Línea Producto,17,0.12,81,P
Família Producto,43,0.30,94,PE


In [18]:
df_features.to_parquet("res//procesed/df_features.parquet")

Apenas modificamos el dataset, ya que cada modelo precisara de su propio feature engeniering. 